## Mini Project: Dry Bean Type Classification

#### Objective
An agriculture company approaches you with a challenge: they are currently classifying different types of dry beans manually, a process that is labour-intensive, prone to errors and inefficient at scale. They want to automate this classification process using Artificial Intelligence to improve accuracy, reduce operational costs and ensure consistent quality in packaging and distribution.
As a data scientist, your role is to help them build a machine learning solution that can accurately classify bean types based on physical characteristics such as area, perimeter, shape and compactness. 

By doing this, you'll:

•	Automate the bean classification process using supervised learning techniques

•	Help the company reduce manual labour and cost

•	Improve the speed and reliability of quality control operations

•	Deliver a scalable solution for real-time classification in industrial settings

This project not only enhances your understanding of supervised machine learning but also demonstrates how AI can be directly applied to Agri-tech and food processing industries for tangible business impact.


#### 1. Import and Load the Data

•	Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn etc.)

•	Load the dataset and explore it using .head(), .info(), and .describe()


#!pip install xgboost

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, PowerTransformer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import boxcox

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)



# Importing dataset
bean_df = pd.read_excel("Beans_Multiclass_Classification.xlsx")
bean_df.head()
bean_df.info()
bean_df.describe()

#view Duplicates , missing values , skewness
df_num = bean_df.select_dtypes(include='number')
print(f"Dupicates \n {bean_df.duplicated().sum()}\n")
print(f"Missings \n {bean_df.isnull().sum().sort_values(ascending =False)}\n")
# ================================== Remove duplicates
bean_df = bean_df.drop_duplicates()
print(f"Dupicates \n {bean_df.duplicated().sum()}\n")


Insight: 

The dataset contains 13,611 observations and 16 numerical features. Since all variables are numeric, no categorical encoding is required.

The dataset exhibits scale imbalance, skewness, outliers, and multicollinearity, but also contains highly informative geometric features. With appropriate preprocessing and feature selection, it is well-suited for building robust machine learning models, particularly using tree-based algorithms.

### 2. Exploratory Data Analysis (EDA)

•	Visualize distributions of features using histograms and boxplots
•	Analyze the class distribution (check for class imbalance)
•	Plot feature correlations (eg heatmap)
•	Visualize multivariate relationships (pairplot)
•	Summarize key findings



# split the target and independent variables
df_target = bean_df['Class']
df_ind_var = bean_df.drop(columns = ['Class'])

# Analyze the class distribution (check for class imbalance)
bean_df['Class'].value_counts(normalize = True)
print(f"skewness \n {df_num.skew().sort_values(ascending =False)}")

# Visualize distributions of features using histograms and boxplots
#==============================================================
# function - Numeric variables boxplot and historgram to see distribution
#==============================================================

def analyze_numeric_distribution(df, num_cols):

    for col in num_cols:
        
        skew_val = df[col].skew()
        
        fig, ax = plt.subplots(1,2, figsize=(12,4))
        
        # Histogram
        sns.histplot(df[col], kde=True, ax=ax[0])
        ax[0].set_title(f"{col} Distribution\nSkewness = {round(skew_val,2)}")
        
        # Boxplot
        sns.boxplot(x=df[col], ax=ax[1])
        ax[1].set_title(f"{col} Boxplot")
        
        plt.show()
        
num_cols = bean_df.select_dtypes(include='number').columns

analyze_numeric_distribution(df_ind_var, num_cols)


#=========================================================================
# relationship between variables
#=====================================================================
#1. Numeric to Numeric

# visualize using heatmap
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt


df_num = bean_df.select_dtypes(include=['number'])

print(df_num.corr())

plt.figure(figsize=(12,8))
sns.heatmap(df_num.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.show()

sns.pairplot(df_num)
plt.show()

Insight : 
1. Strong positive correlation exists among size-related features, indicating redundancy
2. Shape-related features are highly interdependent and show both positive and negative correlations
3. Significant multicollinearity is present in the dataset
4. Some features like Extent provide independent information
5. Pairplot reveals both linear and non-linear relationships, suggesting complex feature interactions
6. The dataset is well-suited for tree-based models rather than linear models
7. From heatmap, Area and ConvexArea is Same we can drop one, some variables are highly correlated with each other. 

3. Missing Values & Outlier Treatment
    •	Check for and handle missing values
    •	Detect and treat outliers if needed (Z-score / IQR methods/boxplots)

print(f"Missings \n {bean_df.isnull().sum().sort_values(ascending =False)}\n")


def dynamic_outlier_treatment(df, method="iqr", cap=True):

    df_out = df.copy()
    report = []

    numeric_cols = df.select_dtypes(include=['int64','float64']).columns

    for col in numeric_cols:

        if method == "iqr":

            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers = df[(df[col] < lower) | (df[col] > upper)][col]

            if len(outliers) == 0:
                action = "No Outliers"

            else:

                if cap:
                    df_out[col] = np.where(df[col] > upper, upper, df[col])
                    df_out[col] = np.where(df_out[col] < lower, lower, df_out[col])
                    action = "Outliers Capped (IQR)"

                else:
                    df_out = df_out[(df_out[col] >= lower) & (df_out[col] <= upper)]
                    action = "Outliers Removed"

            report.append({
                "Column": col,
                "Method": "IQR",
                "Lower Bound": lower,
                "Upper Bound": upper,
                "Outlier Count": len(outliers),
                "Action": action
            })

        elif method == "zscore":

            mean = df[col].mean()
            std = df[col].std()

            lower = mean - 3 * std
            upper = mean + 3 * std

            z = (df[col] - mean) / std

            outliers = df[abs(z) > 3][col]

            if len(outliers) == 0:
                action = "No Outliers"

            else:

                if cap:
                    df_out[col] = np.where(df[col] > upper, upper, df[col])
                    df_out[col] = np.where(df_out[col] < lower, lower, df_out[col])
                    action = "Outliers Capped (Z-score)"

                else:
                    df_out = df_out[abs(z) <= 3]
                    action = "Outliers Removed"

            report.append({
                "Column": col,
                "Method": "Z-score",
                "Lower Bound": lower,
                "Upper Bound": upper,
                "Outlier Count": len(outliers),
                "Action": action
            })

    report_df = pd.DataFrame(report)

    return df_out, report_df
df1, outlier_report = dynamic_outlier_treatment(df_num) 

outlier_report

Feature Engineering & Preprocessing
•	Scale numerical features (StandardScaler / MinMaxScaler)
•	Encode categorical variables if necessary
•	Check and treat skewness if required
•	Split data into train/test sets (use stratified sampling while splitting)

def dynamic_skew_transform(df, skew_threshold=1):

    df_transformed = df.copy()
    report = []

    numeric_cols = df.select_dtypes(include=['int64','float64']).columns

    for col in numeric_cols:

        skew_val = df[col].skew()

        if abs(skew_val) <= skew_threshold:
            report.append({"Column": col, "Skew": skew_val, "Transformation": "None"})
            continue

        series = df[col].replace([np.inf, -np.inf], np.nan)

        # Case 1 : negative values exist
        if (series < 0).any():

            pt = PowerTransformer(method='yeo-johnson')
            df_transformed[col] = pt.fit_transform(series.values.reshape(-1,1))

            method = "Yeo-Johnson"

        # Case 2 : zero values exist
        elif (series == 0).any():

            df_transformed[col] = np.log1p(series)

            method = "Log1p"

        # Case 3 : strictly positive
        else:

            valid_index = series.dropna().index
            transformed, _ = boxcox(series.loc[valid_index])

            df_transformed.loc[valid_index, col] = transformed

            method = "Box-Cox"

        report.append({
            "Column": col,
            "Skew": skew_val,
            "Transformation": method
        })

    report_df = pd.DataFrame(report)

    return df_transformed, report_df

#================================================================================================
# Scaling 
#================================================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

def dynamic_scaling(df, skew_threshold=1):

    df_scaled = df.copy()
    report = []

    numeric_cols = df.select_dtypes(include=['int64','float64']).columns

    for col in numeric_cols:

        skew_val = df[col].skew()

        # Detect outliers using IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers_exist = ((df[col] < lower) | (df[col] > upper)).any()

        # Choose scaler dynamically
        if outliers_exist:
            scaler = RobustScaler()
            method = "RobustScaler"

        elif abs(skew_val) > skew_threshold:
            scaler = MinMaxScaler()
            method = "MinMaxScaler"

        else:
            scaler = StandardScaler()
            method = "StandardScaler"

        df_scaled[col] = scaler.fit_transform(df[[col]])

        report.append({
            "Column": col,
            "Skew": skew_val,
            "Outliers": outliers_exist,
            "Scaler": method
        })

    report_df = pd.DataFrame(report)

    return df_scaled, report_df


# calling skewness and scaling functions which created above
df2, skew_report = dynamic_skew_transform(df1)
df3, scaling_report = dynamic_scaling(df2)

print(df3.shape)
print(len(df_target))

df_dv = bean_df['Class']
df_iv = df3

# encoding target variable  # to avoid errors in XGboost

le = LabelEncoder()
df_dv_encoded = le.fit_transform(df_dv)


#	Split data into train/test sets (use stratified sampling while splitting)
X_train, X_test, y_train, y_test = train_test_split(df_iv,df_dv_encoded, test_size =0.2, random_state = 42,stratify=df_dv_encoded)

#### 5. Model Building: Try Multiple Classifiers
Train and test the dataset on a variety of supervised classification algorithms:
•	Logistic Regression
•	Decision Tree Classifier
•	Random Forest Classifier
•	K-Nearest Neighbors (KNN)
•	Support Vector Machine (SVM)
•	Ensemble Learning Methods
•	Naive Bayes and more…


models = {

    "Logistic Regression": LogisticRegression(max_iter=500),

    "KNN": KNeighborsClassifier(),

    "SVM": SVC(),

    "Naive Bayes": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(),

    "Random Forest": RandomForestClassifier(),

    "Gradient Boosting": GradientBoostingClassifier(),

    "AdaBoost": AdaBoostClassifier(),

    "SGD Classifier": SGDClassifier(),

    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")

}


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    # Precision, Recall, F1 (use 'weighted' for multiclass)
    train_precision = precision_score(y_train, y_train_pred, average='weighted')
    test_precision = precision_score(y_test, y_test_pred, average='weighted')

    train_recall = recall_score(y_train, y_train_pred, average='weighted')
    test_recall = recall_score(y_test, y_test_pred, average='weighted')

    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')

    # Append results
    results.append([
        name,
        train_acc, test_acc,
        train_precision, test_precision,
        train_recall, test_recall,
        train_f1, test_f1
    ])

# Create DataFrame
results_df_stratify = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train Accuracy", "Test Accuracy",
        "Train Precision", "Test Precision",
        "Train Recall", "Test Recall",
        "Train F1 Score", "Test F1 Score"
    ]
)

# Sort by Test Accuracy
results_df_stratify = results_df_stratify.sort_values(
    by="Test Accuracy",
    ascending=False
).round(4)

print(results_df_stratify)



# Define Stratified K-Fold
from sklearn.model_selection import cross_val_score, StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Apply Cross Validation to All Models
cv_results = []

for name, model in models.items():
    
    # Accuracy
    acc_scores = cross_val_score(model, df_iv, df_dv_encoded, 
                                cv=skf, scoring='accuracy')
    
    # F1 Score (better for classification)
    f1_scores = cross_val_score(model, df_iv, df_dv_encoded, 
                               cv=skf, scoring='f1_weighted')
    
    cv_results.append([
        name,
        acc_scores.mean(),
        acc_scores.std(),
        f1_scores.mean(),
        f1_scores.std()
    ])

# Create DataFrame
cv_results_df = pd.DataFrame(
    cv_results,
    columns=[
        "Model",
        "CV Accuracy Mean",
        "CV Accuracy Std",
        "CV F1 Mean",
        "CV F1 Std"
    ]
)

# Sort by best F1 score
cv_results_df = cv_results_df.sort_values(
    by="CV F1 Mean",
    ascending=False
).round(4)

print(cv_results_df)

# imbalance Handling

# data is moderately imbalance so
# Handling Class Imbalance
•	Apply techniques like:
o	SMOTE (Synthetic Minority Over-sampling)
o	Random Oversampling / Undersampling
o	Class weighting


# SMOTE - Creates synthetic samples for minority classes
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(df_iv, df_dv_encoded)

# Random Oversampling - Simply duplicates minority samples
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(df_iv, df_dv_encoded)

# Random Undersampling - Reduces majority class (Risk: Loss of important data)
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(df_iv, df_dv_encoded)

# Class Weighting (Very Important & Simple) - Best option for models like:Logistic Regression,Random Forest,XGBoost
model = XGBClassifier(scale_pos_weight=1)  # for binary

# For sklearn models:
model_RF = RandomForestClassifier(class_weight='balanced')

Model Evaluation & Overfitting Check
Use appropriate classification metrics:
•	Accuracy
•	Precision, Recall, and F1-Score (for each class)
•	Confusion Matrix

Check for Overfitting:
Compare training vs test accuracy and performance metrics.


from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import pandas as pd

evaluation_results = []

for name, model in models.items():
    
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    # Precision, Recall, F1 (weighted for multiclass)
    train_precision = precision_score(y_train, y_train_pred, average='weighted')
    test_precision = precision_score(y_test, y_test_pred, average='weighted')
    
    train_recall = recall_score(y_train, y_train_pred, average='weighted')
    test_recall = recall_score(y_test, y_test_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Store results
    evaluation_results.append([
        name,
        train_acc, test_acc,
        train_precision, test_precision,
        train_recall, test_recall,
        train_f1, test_f1
    ])

# Create DataFrame
evaluation_df = pd.DataFrame(
    evaluation_results,
    columns=[
        "Model",
        "Train Accuracy", "Test Accuracy",
        "Train Precision", "Test Precision",
        "Train Recall", "Test Recall",
        "Train F1 Score", "Test F1 Score"
    ]
)

evaluation_df = evaluation_df.round(4).sort_values(by="Test F1 Score", ascending=False)

print(evaluation_df)

# detailed classification report - per class

for name, model in models.items():
    print(f"\n🔹 Model: {name}")
    
    y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred))

# Confusion matrix

import seaborn as sns
import matplotlib.pyplot as plt

for name, model in models.items():
    
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    

# Automatic Overfitting Detection

evaluation_df["Overfitting Gap"] = (
    evaluation_df["Train Accuracy"] - evaluation_df["Test Accuracy"]
)

print(evaluation_df)

Hyperparameter Tuning
•	Use GridSearchCV or RandomizedSearchCV to optimize parameters for top-performing models
•	Document the best parameters and performance improvement



# Select Top Models

top_models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
}

# 2. GridSearchCV (Exhaustive Search)
from sklearn.model_selection import GridSearchCV

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print("Best RF Params:", rf_grid.best_params_)
print("Best RF Score:", rf_grid.best_score_)

xgb_params = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    param_grid=xgb_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print("Best XGB Params:", xgb_grid.best_params_)
print("Best XGB Score:", xgb_grid.best_score_)



# RandomizedSearchCV (Faster Alternative)
from sklearn.model_selection import RandomizedSearchCV

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    param_distributions=xgb_params,
    n_iter=10,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

xgb_random.fit(X_train, y_train)

print("Best Params (Random Search):", xgb_random.best_params_)

# Evaluate Tuned Model

best_model = xgb_grid.best_estimator_

y_pred = best_model.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

# Compare Before vs After Tuning
before_score = evaluation_df[evaluation_df["Model"]=="XGBoost"]["Test F1 Score"].values[0]
after_score = xgb_grid.best_score_

print("Before Tuning F1:", before_score)
print("After Tuning F1:", after_score)


## . Model Comparison Table

from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

comparison_results = []

for name, model in models.items():
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    # F1 Score (weighted for multiclass)
    f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Overfitting check
    gap = train_acc - test_acc
    
    if gap > 0.05:
        overfit = "Yes"
    else:
        overfit = "No"
    
    # Store results
    comparison_results.append([
        name,
        train_acc,
        test_acc,
        f1,
        overfit
    ])

# Create DataFrame
comparison_df = pd.DataFrame(
    comparison_results,
    columns=[
        "Model",
        "Train Accuracy",
        "Test Accuracy",
        "F1 Score",
        "Overfitting (Y/N)"
    ]
)

# Sort by best model
comparison_df = comparison_df.sort_values(
    by="F1 Score",
    ascending=False
).round(4)

print(comparison_df)

best_model = comparison_df.iloc[0]
print("\nBest Model:")
print(best_model)